# The whole tree — all levels + visual export

Continues `tree-level1.ipynb` (which built the 41 level-1 nodes and probed 4 summaries):
1. summarize every remaining level-1 node (map-reduce recipe, all DB-persisted &
   content-addressed — reruns only fill gaps, so this cell is interruption-safe);
2. build **level 2**: embed the level-1 summaries, cluster them (UMAP-10 seed 42 +
   HDBSCAN leaf mcs=3 + nearest-assignment — the frozen recipe scaled down to 41 points),
   summarize parents with the `level2` composition prompt over child summaries;
3. one **root** node over the level-2 parents (same prompt);
4. export `.hrono/tree.html` — self-contained, every node expandable: metadata
   (level/size/coherence/model/cost/date), rendered summary, and for leaves the chunks
   with their condensation and raw source, each expandable.

In [1]:
import json
import sqlite3
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path

import numpy as np

import summarize_experiments as se

APP_DIR = Path.cwd()
DB_PATH = APP_DIR / ".hrono" / "hrono-v2.db"
GIT_CACHE = APP_DIR / ".hrono" / "cache" / "git"
MODEL_NAME = "ibm-granite/granite-embedding-97m-multilingual-r2"

db = sqlite3.connect(DB_PATH)


def utcnow():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def prompt_id(name, text):
    with db:
        db.execute("INSERT OR IGNORE INTO prompts (name, text) VALUES (?,?)", (name, text))
    return db.execute("SELECT id FROM prompts WHERE name = ? AND text = ?", (name, text)).fetchone()[0]


MAP_PROMPT_ID = db.execute("SELECT id FROM prompts WHERE name='map-condense'").fetchone()[0]
REDUCE_PROMPT_ID = db.execute("SELECT id FROM prompts WHERE name='adaptive-gotchas'").fetchone()[0]
# level-2+ prompt: the round-7 winner from summarize-level2-experiments.ipynb,
# read from the prompts table — the DB is the single source of truth for prompt text
L2_PROMPT_ID, L2_PROMPT = db.execute(
    "SELECT id, text FROM prompts WHERE name = 'v7b-sized'").fetchone()

# level-1 reduce prompt: adaptive-gotchas purged of tree/node vocabulary (the model
# must not know it is inside a tree — same principle as level 2)
L1_PROMPT = se.instructions("adaptive-gotchas").replace(
    """The summary is a node in a
knowledge tree: a reader (human or AI agent) uses it to understand what this group
is and contains, without opening the chunks.""",
    """A reader (human or AI agent)
uses it to understand what this group is and contains, without opening the chunks.""")
assert "node" not in L1_PROMPT and "tree" not in L1_PROMPT
L1_PROMPT_ID = prompt_id("l1-pure-adaptive-gotchas", L1_PROMPT)


def claude_call(prompt, model="haiku"):
    # se.call_claude uses asyncio.run — illegal on Jupyter's loop thread, so hop threads
    with ThreadPoolExecutor(max_workers=1) as ex:
        return ex.submit(se.call_claude, prompt, model).result()


@dataclass
class C:  # duck-typed stand-in for summarize_experiments.Chunk
    label: str
    hash: str
    text: str


def condense_all(chunks):
    out, missing = {}, []
    for c in chunks:
        row = db.execute("SELECT text FROM condensations WHERE hash = ?", (c.hash,)).fetchone()
        if row:
            out[c.label] = row[0]
        else:
            missing.append(c)
    if missing:
        with ThreadPoolExecutor(max_workers=se.PARALLEL_CALLS) as pool:
            results = list(pool.map(se.condense_chunk, missing))
        out.update({c.label: t for c, t in zip(missing, results)})
        with db:
            db.executemany(
                "INSERT OR IGNORE INTO condensations (hash, model, text, prompt_id, created_at)"
                " VALUES (?,?,?,?,?)",
                [(c.hash, se.MAP_MODEL, t, MAP_PROMPT_ID, utcnow()) for c, t in zip(missing, results)])
    return out


def chunks_of_node(node_id):
    out = []
    for path, chunk_no, h, inst, s, e in db.execute("""
            SELECT c.path, c.chunk_no, c.hash, c.instance, c.byte_start, c.byte_end
            FROM edges e JOIN chunks c ON c.id = e.child_id
            WHERE e.parent_id = ? AND e.child_kind = 'chunk'
            ORDER BY c.path, c.chunk_no""", (node_id,)):
        raw = (GIT_CACHE / inst / path).read_bytes()
        out.append(C(label=f"{path}#chunk-{chunk_no}", hash=h,
                     text=raw[s:e].decode("utf-8", "replace")))
    return out

## 1. Summarize every remaining level-1 node

Same probe recipe, all 41 nodes. Interruption-safe: condensations and summaries land in
the DB as they complete; a rerun skips everything done.

In [2]:
# prompt change invalidates: level-1 summaries made with any other prompt are stale,
# and everything above them too (their inputs change)
stale = db.execute("SELECT COUNT(*) FROM nodes WHERE level = 1 AND summary IS NOT NULL"
                   " AND (summary_prompt_id IS NULL OR summary_prompt_id != ?)",
                   (L1_PROMPT_ID,)).fetchone()[0]
if stale:
    with db:
        db.execute("UPDATE nodes SET summary=NULL, summary_model=NULL, summary_cost_usd=NULL,"
                   " summary_prompt_id=NULL, summary_created_at=NULL WHERE level = 1"
                   " AND (summary_prompt_id IS NULL OR summary_prompt_id != ?)", (L1_PROMPT_ID,))
        db.execute("UPDATE nodes SET summary=NULL, summary_model=NULL, summary_cost_usd=NULL,"
                   " summary_prompt_id=NULL, summary_created_at=NULL WHERE level >= 2")
    print(f"invalidated {stale} level-1 summaries (prompt change) and all levels above")

todo = [r[0] for r in db.execute(
    "SELECT id FROM nodes WHERE level = 1 AND summary IS NULL ORDER BY id")]
print(f"{len(todo)} level-1 nodes to summarize")
for n, node_id in enumerate(todo, 1):
    chunks = chunks_of_node(node_id)
    condensed = condense_all(chunks)
    content = se.render_condensed(chunks, condensed)
    result = claude_call(L1_PROMPT + "\n\n" + content, "haiku")
    with db:
        db.execute("UPDATE nodes SET summary = ?, summary_model = ?, summary_cost_usd = ?,"
                   " summary_prompt_id = ?, summary_created_at = ? WHERE id = ?",
                   (result["text"], "haiku:l1-pure-gotchas-condensed", result["cost_usd"],
                    L1_PROMPT_ID, utcnow(), node_id))
    print(f"  [{n}/{len(todo)}] node {node_id}: {len(chunks)} chunks, ${result['cost_usd']:.3f}")

count, cost = db.execute(
    "SELECT COUNT(*), SUM(summary_cost_usd) FROM nodes WHERE level = 1 AND summary IS NOT NULL"
).fetchone()
print(f"level 1 complete: {count} summaries, total ${cost:.2f}")

0 level-1 nodes to summarize
level 1 complete: 41 summaries, total $1.00


## 2. Level 2 — cluster the level-1 summaries, summarize the parents

Node summary embeddings (granite-97m, content only — nodes have no path), the settled
pipeline scaled to 41 points (UMAP-10, HDBSCAN leaf mcs=3, noise assigned to nearest
clustered node in the original space). Parent summaries use the `level2` prompt — the
round-5 composition test, now on real children. Summary reuse by member-set signature,
same as level 1.

In [3]:
import umap
from sentence_transformers import SentenceTransformer
from sklearn.cluster import HDBSCAN

l1 = db.execute("SELECT id, sig, summary FROM nodes WHERE level = 1 ORDER BY id").fetchall()
emb_model = SentenceTransformer(MODEL_NAME)
vecs = emb_model.encode([s for _, _, s in l1], batch_size=16,
                        normalize_embeddings=True, show_progress_bar=False)

reduced = umap.UMAP(n_components=10, metric="cosine", random_state=42).fit_transform(vecs)
labels = HDBSCAN(min_cluster_size=3, metric="euclidean",
                 cluster_selection_method="leaf").fit_predict(reduced)
noise = np.where(labels == -1)[0]
core = np.where(labels >= 0)[0]
if len(noise) and len(core):
    sims = vecs[noise] @ vecs[core].T
    labels[noise] = labels[core[sims.argmax(axis=1)]]
print(f"level 2: {labels.max() + 1} parents over {len(l1)} children ({len(noise)} assigned)")

saved = {r[0]: r[1:] for r in db.execute(
    "SELECT sig, summary, summary_model, summary_cost_usd, summary_prompt_id,"
    " summary_created_at FROM nodes WHERE level >= 2 AND summary IS NOT NULL"
    " AND summary_prompt_id = ?", (L2_PROMPT_ID,))}  # prompt change invalidates reuse
with db:
    db.execute("DELETE FROM edges WHERE child_kind = 'node'")
    db.execute("DELETE FROM nodes WHERE level >= 2")

l2_ids = []
with db:
    for label in sorted(set(labels)):
        idx = [i for i, l in enumerate(labels) if l == label]
        sig = sha256("".join(sorted(l1[i][1] for i in idx)).encode()).hexdigest()
        coherence = float((vecs[idx] @ vecs[idx].T).mean()) if len(idx) > 1 else 1.0
        summary, s_model, s_cost, s_prompt, s_created = saved.get(sig, (None,) * 5)
        cur = db.execute(
            "INSERT INTO nodes (level, size, coherence, sig, summary, summary_model,"
            " summary_cost_usd, summary_prompt_id, summary_created_at)"
            " VALUES (2, ?, ?, ?, ?, ?, ?, ?, ?)",
            (len(idx), round(coherence, 4), sig, summary, s_model, s_cost, s_prompt, s_created))
        db.executemany("INSERT INTO edges (parent_id, child_kind, child_id, assigned)"
                       " VALUES (?, 'node', ?, ?)",
                       [(cur.lastrowid, l1[i][0], int(i in noise)) for i in idx])
        l2_ids.append(cur.lastrowid)


def summarize_over_children(node_id):
    kids = db.execute("""
        SELECT n.id, n.summary FROM edges e JOIN nodes n ON n.id = e.child_id
        WHERE e.parent_id = ? AND e.child_kind = 'node' ORDER BY n.id""", (node_id,)).fetchall()
    # children rendered as pure titled texts — the model never learns about trees/nodes
    content = "\n\n".join(
        f"=== {(s or '').lstrip('# ').splitlines()[0][:90]} ===\n{s}" for _, s in kids)
    result = claude_call(L2_PROMPT + "\n\n" + content, "sonnet")
    with db:
        db.execute("UPDATE nodes SET summary = ?, summary_model = ?, summary_cost_usd = ?,"
                   " summary_prompt_id = ?, summary_created_at = ? WHERE id = ?",
                   (result["text"], "sonnet:l2-pure-v7b", result["cost_usd"],
                    L2_PROMPT_ID, utcnow(), node_id))
    print(f"  node {node_id}: {len(kids)} children, ${result['cost_usd']:.3f}")


for nid in l2_ids:
    if not db.execute("SELECT summary FROM nodes WHERE id = ?", (nid,)).fetchone()[0]:
        summarize_over_children(nid)

Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


level 2: 6 parents over 41 children (3 assigned)


/Users/orest/repos/rad/hrono/app/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


## 3. Root — one node over the level-2 parents

In [4]:
if len(l2_ids) > 1:
    sigs = [db.execute("SELECT sig FROM nodes WHERE id = ?", (n,)).fetchone()[0] for n in l2_ids]
    sig = sha256("".join(sorted(sigs)).encode()).hexdigest()
    summary, s_model, s_cost, s_prompt, s_created = saved.get(sig, (None,) * 5)
    with db:
        cur = db.execute(
            "INSERT INTO nodes (level, size, coherence, sig, summary, summary_model,"
            " summary_cost_usd, summary_prompt_id, summary_created_at)"
            " VALUES (3, ?, 1.0, ?, ?, ?, ?, ?, ?)",
            (len(l2_ids), sig, summary, s_model, s_cost, s_prompt, s_created))
        root_id = cur.lastrowid
        db.executemany("INSERT INTO edges (parent_id, child_kind, child_id, assigned)"
                       " VALUES (?, 'node', ?, 0)", [(root_id, n) for n in l2_ids])
    if not summary:
        summarize_over_children(root_id)
else:
    root_id = l2_ids[0]
print(f"root: node {root_id}")

root: node 48


## 4. Visual export — `.hrono/tree.html`

Self-contained page, nothing external. Nested `<details>`: node header carries the meta
(level, size, coherence, model, cost, date); body holds the rendered summary; leaf nodes
list their chunks, each expandable into struct paths, the Haiku condensation, and the raw
source bytes.

In [5]:
import html as H

import markdown as md_lib

CSS = """
body { font: 14px/1.5 -apple-system, sans-serif; margin: 20px auto; max-width: 1100px; color: #222; }
details { margin: 4px 0 4px 14px; border-left: 2px solid #ddd; padding-left: 10px; }
summary { cursor: pointer; }
summary:hover { background: #f6f8fa; }
.node > summary { font-weight: 600; }
.meta { color: #667; font-size: 12px; font-weight: 400; }
.sum { background: #fafbfc; border: 1px solid #eee; border-radius: 6px; padding: 4px 14px; margin: 6px 0; }
.sum h1, .sum h2, .sum h3 { font-size: 15px; margin: 10px 0 4px; }
.chunk > summary { font-family: monospace; font-size: 12px; }
pre { background: #f6f8fa; border-radius: 6px; padding: 8px; overflow-x: auto; font-size: 12px; white-space: pre-wrap; }
.badge { background: #eef2ff; border-radius: 8px; padding: 0 7px; font-size: 11px; margin-left: 5px; }
"""


def md(text):
    return md_lib.markdown(text or "*(no summary)*")


def chunk_html(cid, path, chunk_no, h, inst, s, e, sp):
    raw = (GIT_CACHE / inst / path).read_bytes()[s:e].decode("utf-8", "replace")
    cond = db.execute("SELECT text FROM condensations WHERE hash = ?", (h,)).fetchone()
    cond_html = (f"<details><summary>condensation (haiku)</summary><pre>{H.escape(cond[0])}</pre></details>"
                 if cond else "")
    return (f'<details class="chunk"><summary>{H.escape(path)}#chunk-{chunk_no}'
            f'<span class="badge">{e - s} B</span></summary>'
            f'<div class="meta">struct: {H.escape(sp)}<br>hash {h[:16]}…</div>'
            f'{cond_html}'
            f'<details><summary>source</summary><pre>{H.escape(raw)}</pre></details></details>')


def node_html(node_id):
    level, size, coh, summary, s_model, s_cost, s_created = db.execute(
        "SELECT level, size, coherence, summary, summary_model, summary_cost_usd,"
        " summary_created_at FROM nodes WHERE id = ?", (node_id,)).fetchone()
    title = (summary or "").lstrip("# ").split("\n")[0][:90] or f"node {node_id}"
    meta = (f'<span class="meta">L{level} · {size} members · coherence {coh:.3f}'
            + (f' · {s_model} · ${s_cost:.3f} · {s_created}' if s_model else '') + '</span>')
    parts = [f'<details class="node"><summary>#{node_id} {H.escape(title)} {meta}</summary>',
             f'<div class="sum">{md(summary)}</div>']
    kids = db.execute("SELECT child_kind, child_id FROM edges WHERE parent_id = ?"
                      " ORDER BY child_kind, child_id", (node_id,)).fetchall()
    for kind, cid in kids:
        if kind == "node":
            parts.append(node_html(cid))
        else:
            row = db.execute("SELECT path, chunk_no, hash, instance, byte_start, byte_end,"
                             " struct_uris FROM chunks WHERE id = ?", (cid,)).fetchone()
            parts.append(chunk_html(cid, *row))
    parts.append("</details>")
    return "".join(parts)


stats = db.execute("SELECT COUNT(*), SUM(summary_cost_usd) FROM nodes WHERE summary IS NOT NULL").fetchone()
page = (f"<!doctype html><meta charset='utf-8'><title>hrono tree</title><style>{CSS}</style>"
        f"<h1>hrono — knowledge tree</h1>"
        f"<p class='meta'>{stats[0]} summarized nodes · total LLM cost ${stats[1]:.2f} · "
        f"corpus: 501 chunks · generated {utcnow()}</p>"
        + node_html(root_id))
out_path = APP_DIR / ".hrono" / "tree.html"
out_path.write_text(page)
print(f"wrote {out_path} ({out_path.stat().st_size / 1e6:.1f} MB) — open in a browser")

wrote /Users/orest/repos/rad/hrono/app/.hrono/tree.html (1.9 MB) — open in a browser


## Faithfulness gate — mechanical, no LLM

Every identifier-like token in every generated text must literally exist in the text's
source: condensations vs raw chunk bytes, level-1 summaries vs their chunks' raw bytes,
level-2+ summaries vs their children's summaries. Reuses the experiments' identifier
regex; simpler matching than check_faithfulness.py (case-insensitive containment), so
treat hits as SUSPECTS to eyeball, not verdicts.

In [6]:
from check_faithfulness import IDENTIFIER_RE


def suspects(generated, source):
    """check_faithfulness rules replicated: case-insensitive containment, chunk labels
    count as source, slash/dot-compounds pass if every part exists, e.g-style regex
    artifacts dropped."""
    src = source.lower()
    out = set()
    for m in IDENTIFIER_RE.finditer(generated):
        tok = m.group(0)
        if tok.lower() in src or tok.lower() in ("e.g", "i.e"):
            continue
        parts = [p for p in __import__("re").split(r"[/.]", tok) if p]
        if parts and all(p.lower() in src for p in parts):
            continue
        out.add(tok)
    return sorted(out)


chunk_src, labels_by_hash = {}, {}
for inst, path, s, e, h in db.execute(
        "SELECT instance, path, byte_start, byte_end, hash FROM chunks"):
    if h not in chunk_src:
        chunk_src[h] = (GIT_CACHE / inst / path).read_bytes()[s:e].decode("utf-8", "replace")
    labels_by_hash.setdefault(h, []).append(path)

report = []
for h, text in db.execute("SELECT hash, text FROM condensations"):
    source = chunk_src.get(h, "") + "\n" + "\n".join(labels_by_hash.get(h, []))
    bad = suspects(text, source)
    if bad:
        report.append(("condensation", h[:12], bad))
for nid, in db.execute("SELECT id FROM nodes WHERE level = 1"):
    summary = db.execute("SELECT summary FROM nodes WHERE id = ?", (nid,)).fetchone()[0]
    hashes = [r[0] for r in db.execute(
        "SELECT c.hash FROM edges e JOIN chunks c ON c.id = e.child_id"
        " WHERE e.parent_id = ?", (nid,))]
    source = "\n".join(chunk_src[h] for h in hashes) + "\n" + "\n".join(
        p for h in hashes for p in labels_by_hash[h])
    bad = suspects(summary, source)
    if bad:
        report.append(("L1 summary", f"node {nid}", bad))
for nid, in db.execute("SELECT id FROM nodes WHERE level >= 2"):
    summary = db.execute("SELECT summary FROM nodes WHERE id = ?", (nid,)).fetchone()[0]
    source = "\n".join(r[0] for r in db.execute(
        "SELECT n.summary FROM edges e JOIN nodes n ON n.id = e.child_id"
        " WHERE e.parent_id = ?", (nid,)))
    bad = suspects(summary, source)
    if bad:
        report.append(("L2+ summary", f"node {nid}", bad))

texts_checked = (db.execute("SELECT COUNT(*) FROM condensations").fetchone()[0]
                 + db.execute("SELECT COUNT(*) FROM nodes WHERE summary IS NOT NULL").fetchone()[0])
flagged = len(report)
total_suspects = sum(len(b) for _, _, b in report)
print(f"checked {texts_checked} generated texts; {flagged} flagged, {total_suspects} suspect tokens total")
for kind, ref, bad in sorted(report, key=lambda r: -len(r[2]))[:15]:
    print(f"  {kind:<13} {ref:<14} {len(bad):>3}: {', '.join(bad[:10])}")

checked 549 generated texts; 158 flagged, 280 suspect tokens total
  L1 summary    node 14         13: 303, 400, 403, 404, 405, 406, 409, 415, 500, PhoneNumber
  condensation  77860e0a445e     8: 400, 403, 404, 405, 406, 409, 415, 500
  condensation  3d24ad25eaf6     7: 400, 403, 404, 405, 406, 415, 500
  L1 summary    node 25          6: 204, 401, 403, name/logic, single/batch, unit/integration
  L1 summary    node 41          6: 1234, 555, 567890, Material/Ionicons, VoIP/webphone, digit/non-
  L1 summary    node 22          5: TypeScript, burgundy/rust, localStorage, purple/orange, status/error
  L1 summary    node 39          5: Ringing/pending, TypeScript, VoIP/SIP, backspace/deletion, hold/resume
  condensation  52670a7f461f     4: newHeight, newWidth, originalHeight, originalWidth
  condensation  2fe2a47d3d2b     4: 135, 401, 403, name/logic
  L1 summary    node 1           4: 200, 201, 204, null/existence
  L1 summary    node 6           4: TypeScript, com.scisbo.webphone.{conve

## Observations

(To fill after review of the full tree. Known carried items: level-2 clustering params
are pragmatic (41 points), not the frozen recipe; Sonnet-for-messy heuristic untested;
routing-flavor prompts for upper levels untested.)